<a href="https://colab.research.google.com/github/tzmudder/AAI2026/blob/main/Prompt_Engineering_Part_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os, json, re, textwrap
from datetime import datetime

# Optional: use OpenAI if you have an API key set in Colab secrets or env.
USE_LLM = bool(os.environ.get("OPENAI_API_KEY"))

requests = [
    {
        "request_id": "SR-1001",
        "customer": "Acme Co",
        "channel": "email",
        "message": "Hi team, our invoice export is failing for one user. Not urgent, but we'd like a fix this week."
    },
    {
        "request_id": "SR-1002",
        "customer": "BluePeak",
        "channel": "chat",
        "message": "This is ridiculous. We can’t log in and it’s blocking the entire team from working. Fix it now."
    },
    {
        "request_id": "SR-1003",
        "customer": "Northwind",
        "channel": "phone",
        "message": "We think there may have been unauthorized access. We see suspicious password reset emails. Please respond ASAP."
    },
    {
        "request_id": "SR-1004",
        "customer": "Contoso",
        "channel": "web",
        "message": "Feature request: can you add dark mode? No rush."
    },
]
print(f"Loaded {len(requests)} sample service requests. LLM enabled? {USE_LLM}")

Loaded 4 sample service requests. LLM enabled? False


In [5]:
if USE_LLM:
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    MODEL = "gpt-4.1-mini"  # choose a model your account has

def llm_classify_tone_impact(message: str) -> dict:
    """
    Step 1 in chain: LLM converts text -> structured JSON:
    tone, operational_impact, evidence, and confidence.
    """
    system = (
        "You are an IT service desk triage classifier.\n"
        "Output must be valid JSON only. No markdown.\n"
        "Do not invent internal facts; base judgments only on the message text.\n"
        "Choose from these enums strictly:\n"
        "- tone: [calm, frustrated, angry, panicked]\n"
        "- operational_impact: [low, medium, high, critical]\n"
        "- security_flag: [true, false]\n"
    )

    user = f"""
Classify the service request message.

Return JSON with keys:
- tone
- operational_impact
- security_flag
- reasoning_evidence (array of short phrases quoted or paraphrased from text)
- confidence (0.0 to 1.0)

Message:
\"\"\"{message}\"\"\"
""".strip()

    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    raw = resp.choices[0].message.content
    return json.loads(raw)

In [7]:
TONE_KEYWORDS = {
    "panicked": ["asap", "immediately", "urgent", "right now", "security", "breach", "unauthorized", "suspicious"],
    "angry": ["ridiculous", "unacceptable", "fix it now", "terrible", "angry", "furious"],
    "frustrated": ["can't", "cannot", "won't", "broken", "blocking", "stuck", "issue", "problem", "frustrated"],
}

IMPACT_KEYWORDS = {
    "critical": ["entire team", "all users", "down", "outage", "production", "breach", "unauthorized access"],
    "high": ["blocking", "can't log in", "cannot login", "payments", "orders", "revenue"],
    "medium": ["one user", "some users", "degraded", "slow", "fails"],
    "low": ["feature request", "no rush", "nice to have", "this week", "when you can"],
}

def heuristic_classify_tone_impact(message: str) -> dict:
    msg = message.lower()

    # Tone
    tone = "calm"
    for t in ["panicked", "angry", "frustrated"]:
        if any(k in msg for k in TONE_KEYWORDS[t]):
            tone = t
            break

    # Impact
    operational_impact = "low"
    for lvl in ["critical", "high", "medium", "low"]:
        if any(k in msg for k in IMPACT_KEYWORDS[lvl]):
            operational_impact = lvl
            break

    security_flag = any(k in msg for k in ["unauthorized", "breach", "suspicious", "security", "password reset"])

    evidence = []
    for k in (TONE_KEYWORDS.get(tone, []) + IMPACT_KEYWORDS.get(operational_impact, [])):
        if k in msg:
            evidence.append(k)
    evidence = evidence[:5]

    return {
        "tone": tone,
        "operational_impact": operational_impact,
        "security_flag": bool(security_flag),
        "reasoning_evidence": evidence,
        "confidence": 0.55 if evidence else 0.35
    }

In [8]:
TONE_SCORE = {"calm": 0, "frustrated": 1, "angry": 2, "panicked": 3}
IMPACT_SCORE = {"low": 0, "medium": 2, "high": 4, "critical": 6}

def decide_urgency(tone: str, operational_impact: str, security_flag: bool) -> dict:
    """
    Step 2 in chain: scoring + deterministic urgency mapping.
    """
    score = TONE_SCORE.get(tone, 0) + IMPACT_SCORE.get(operational_impact, 0)

    # Security always bumps urgency
    if security_flag:
        score += 3

    # Map score -> label
    if score >= 9:
        level = "P0 - Critical (Immediate Response)"
        sla = "Respond ≤ 15 min"
    elif score >= 6:
        level = "P1 - High (Urgent)"
        sla = "Respond ≤ 1 hour"
    elif score >= 3:
        level = "P2 - Medium (Standard)"
        sla = "Respond ≤ 1 business day"
    else:
        level = "P3 - Low (Backlog/Planned)"
        sla = "Respond ≤ 3 business days"

    return {"urgency_level": level, "sla_target": sla, "score": score}

In [10]:
def triage_request(req: dict) -> dict:
    # Step 1: classify tone + impact
    if USE_LLM:
        classification = llm_classify_tone_impact(req["message"])
        classification["method"] = "llm"
    else:
        classification = heuristic_classify_tone_impact(req["message"])
        classification["method"] = "heuristic"

    # Step 2: map to urgency
    decision = decide_urgency(
        tone=classification["tone"],
        operational_impact=classification["operational_impact"],
        security_flag=classification["security_flag"]
    )

    return {
        "request_id": req["request_id"],
        "customer": req["customer"],
        "channel": req["channel"],
        "message": req["message"],
        **classification,
        **decision
    }

results = [triage_request(r) for r in requests]

# Create a clean text report
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
lines = []
lines.append("SERVICE DESK TRIAGE REPORT")
lines.append(f"Generated: {now}")
lines.append("=" * 72)

for r in results:
    lines.append(f"Request: {r['request_id']} | Customer: {r['customer']} | Channel: {r['channel']}")
    lines.append(f"Urgency: {r['urgency_level']}  | SLA: {r['sla_target']}  | Score: {r['score']}")
    lines.append(f"Tone: {r['tone']} | Operational Impact: {r['operational_impact']} | Security Flag: {r['security_flag']}")
    if r.get("reasoning_evidence"):
        lines.append("Evidence: " + ", ".join(r["reasoning_evidence"]))
    lines.append("Message:")
    lines.append(textwrap.fill(r["message"], width=72))
    lines.append(f"Classifier: {r.get('method','unknown')} | Confidence: {r.get('confidence','?')}")
    lines.append("-" * 72)

report_text = "\n".join(lines)

out_path = "service_triage_report.txt"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(report_text[:1200])  # preview
print(f"\nSaved report to: {out_path}")

SERVICE DESK TRIAGE REPORT
Generated: 2026-03-01 18:39:33
Request: SR-1001 | Customer: Acme Co | Channel: email
Urgency: P2 - Medium (Standard)  | SLA: Respond ≤ 1 business day  | Score: 5
Tone: panicked | Operational Impact: medium | Security Flag: False
Evidence: urgent, one user
Message:
Hi team, our invoice export is failing for one user. Not urgent, but
we'd like a fix this week.
Classifier: heuristic | Confidence: 0.55
------------------------------------------------------------------------
Request: SR-1002 | Customer: BluePeak | Channel: chat
Urgency: P1 - High (Urgent)  | SLA: Respond ≤ 1 hour  | Score: 8
Tone: angry | Operational Impact: critical | Security Flag: False
Evidence: ridiculous, fix it now, entire team
Message:
This is ridiculous. We can’t log in and it’s blocking the entire team
from working. Fix it now.
Classifier: heuristic | Confidence: 0.55
------------------------------------------------------------------------
Request: SR-1003 | Customer: Northwind | Channel

In [11]:
from google.colab import files
files.download("service_triage_report.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>